# Projection and interpolation

projection problem on $\textbf{x}\in\Omega$

$$u = f(a(\textbf{x}), b(\textbf{x}))$$

variational formulation

$$F(u,v)=\int_\Omega\text{d}\Omega~vu -vf=0$$


## $d=2$ rectangle 

$$\Omega = [0, L_x] \times [0, L_y]$$

$$f(a, b) = a + b$$

In [ ]:
import numpy as np
from ufl.core.expr import Expr
from dolfinx.fem import FunctionSpace

from lucifex.mesh import rectangle_mesh
from lucifex.fem import SpatialFunction as Function, SpatialConstant as Constant
from lucifex.solver import projection_solver, interpolation_solver

def rhs(
    a: Function | Constant | Expr,
    b: Function | Constant | Expr,
) -> Expr:
    f = a + b 
    return f

Lx = 2.0
Ly = 1.0
mesh = rectangle_mesh(Lx, Ly, 20, 10)

a = Constant(mesh, 1.0, name='a')
b = Constant(mesh, 2.0, name='b')

fs = FunctionSpace(mesh, ('P', 1))
uP = Function(fs, name='uP')
u_projector = projection_solver(uP, rhs)(a, b)
uI = Function(fs, name='uI')
u_interpolator = interpolation_solver(uI, rhs)(a, b)

u_projector.solve()
u_interpolator.solve()
expected = a.value + b.value
print(f'Expecting {uP.name} ~= {expected}')
print(np.all(np.isclose(uP.x.array, expected)))
print(f'Expecting {uI.name} ~= {expected}')
print(np.all(np.isclose(uI.x.array, expected)))

print('Mutating a...')
a.value += 1.0
u_projector.solve(overwrite=True)
u_interpolator.solve(overwrite=True)
expected = a.value + b.value
print(f'Now expecting {uP.name} ~= {expected}')
print(np.all(np.isclose(uP.x.array, expected)))
print(f'Now expecting {uI.name} ~= {expected}')
print(np.all(np.isclose(uI.x.array, expected)))

In [ ]:
a = Constant(mesh, 1.0, name='a')
b = Function((mesh, 'P', 1), lambda x: x[0] * x[1], name='b')

uP = Function(fs, name='uP')
u_projector = projection_solver(uP, rhs)(a, b)
uI = Function(fs, name='uI')
u_interpolator = interpolation_solver(uI, rhs)(a, b)

u_projector.solve()
u_interpolator.solve()
expected_min = a.value
expected_max = a.value + Lx * Ly
print(f'Expecting min({uP.name}) ~= {expected_min} and max({uP.name}) ~= {expected_max}')
print(np.isclose(np.min(uP.x.array), expected_min), np.isclose(np.max(uP.x.array), expected_max))
print(f'Expecting min({uI.name}) ~= {expected_min} and max({uI.name}) ~= {expected_max}')
print(np.isclose(np.min(uI.x.array), expected_min), np.isclose(np.max(uI.x.array), expected_max))

print('Mutating b...')
b.interpolate(lambda x: 2 * x[0] * x[1])
u_projector.solve(overwrite=True)
u_interpolator.solve(overwrite=True)
expected_min = a.value
expected_max = a.value + 2 * Lx * Ly
print(f'Now expecting min({uP.name}) ~= {expected_min} and max({uP.name}) ~= {expected_max}')
print(np.isclose(np.min(uP.x.array), expected_min), np.isclose(np.max(uP.x.array), expected_max))
print(f'Now expecting min({uI.name}) ~= {expected_min} and max({uI.name}) ~= {expected_max}')
print(np.isclose(np.min(uI.x.array), expected_min), np.isclose(np.max(uI.x.array), expected_max))